# GraphFrames: finding the influential accounts and communities a support campaign should target in the Twitter network

**Runs on Google Colab.** GraphFrames is a JVM package, which Databricks Free Edition (serverless) cannot load, so this notebook uses local-mode PySpark on Colab. It rebuilds the gold graph tables from the raw CSV with the same functions `GX_Preprocessing` uses on Databricks (`GX_utils.build_graph_silver` / `build_graph_gold`).

Setup: put the repository folder in Google Drive and set `COLAB_REPO_DIR` below to its path.

In [ ]:
import importlib.util
import os
import subprocess
import sys

COLAB_REPO_DIR = "/content/drive/MyDrive/Colab Notebooks/Big Data Analytics/big-data-analysis-project"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    sys.path.insert(0, os.path.join(COLAB_REPO_DIR, "Notebooks"))

# Python API of GraphFrames; the matching JAR is pulled by U.get_spark() (same version, GX_utils.GRAPHFRAMES_VERSION).
packages = ["graphframes-py==0.10.0"] + ([] if importlib.util.find_spec("pyspark") else ["pyspark"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from graphframes import GraphFrame

import GX_utils as U

spark = U.get_spark("GX_GraphFrames")
U.setup_storage(spark)
print("Spark", spark.version, "| Colab:", U.IS_COLAB, "| raw data:", U.RAW_DIR)

# Same Bronze -> Silver -> Gold logic as GX_Preprocessing, materialised once so every algorithm reuses it.
silver_edges, silver_vertices = U.build_graph_silver(U.read_twitter_graph_raw(spark))
gold_graph_edges, gold_account_features = U.build_graph_gold(silver_edges, silver_vertices)
U.write_table(gold_graph_edges, "gold", "graph_edges")
U.write_table(gold_account_features, "gold", "account_features")

vertices = U.read_table(spark, "gold", "account_features")
edges = U.read_table(spark, "gold", "graph_edges")
graph = GraphFrame(vertices, edges)

n_accounts, n_edges = graph.vertices.count(), graph.edges.count()
print(f"accounts: {n_accounts:,} | weighted edges: {n_edges:,}")

## 1. Degree centrality: who receives attention and who broadcasts

In [ ]:
degrees = graph.inDegrees.join(graph.outDegrees, "id", "full").fillna(0)
degrees.select("inDegree", "outDegree").summary("mean", "50%", "90%", "99%", "max").show()

(degrees.join(vertices.select("id", "followers"), "id")
    .orderBy(F.desc("inDegree"))
    .show(10, truncate=False))

## 2. PageRank: influence that accounts for who interacts with you, not only how many

In [ ]:
pagerank = graph.pageRank(resetProbability=0.15, maxIter=10)
ranked = U.maybe_cache(pagerank.vertices.select("id", "followers", "in_degree", "out_degree", "pagerank"))

ranked.orderBy(F.desc("pagerank")).show(10, truncate=False)

(ranked.filter(F.col("followers").isNotNull())
    .agg(F.corr(F.log1p("followers"), F.log1p("pagerank")).alias("corr_log_followers_vs_log_pagerank"),
         F.corr(F.log1p("in_degree"), F.log1p("pagerank")).alias("corr_log_in_degree_vs_log_pagerank"))
    .show())

In [ ]:
sample_pd = (ranked.filter(F.col("followers") > 0)
    .sample(fraction=0.05, seed=U.SEED)
    .select("followers", "pagerank").toPandas())

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(sample_pd["followers"], sample_pd["pagerank"], s=4, alpha=0.4)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Followers"); ax.set_ylabel("PageRank")
ax.set_title("Audience size vs influence in the conversation (5% sample)", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Connected components: how fragmented is the network?

In [ ]:
if not U.IS_DATABRICKS:
    spark.sparkContext.setCheckpointDir(U.work_path("checkpoints", "graphframes"))

components = graph.connectedComponents()
component_sizes = components.groupBy("component").count().orderBy(F.desc("count"))
component_sizes.show(5)

giant = component_sizes.first()["count"]
print(f"components: {component_sizes.count():,} | largest component: {giant:,} accounts ({giant / n_accounts:.1%})")

## 4. Label propagation: communities of accounts that interact mostly with each other

In [ ]:
communities = graph.labelPropagation(maxIter=5).select("id", F.col("label").alias("community"))
community_sizes = communities.groupBy("community").count().orderBy(F.desc("count"))
print(f"communities: {community_sizes.count():,}")
community_sizes.show(10)

In [ ]:
account_graph_metrics = (ranked
    .join(components.select("id", "component"), "id")
    .join(communities, "id"))
U.write_table(account_graph_metrics, "gold", "account_graph_metrics")
account_graph_metrics = U.read_table(spark, "gold", "account_graph_metrics")

top_communities = [r["community"] for r in community_sizes.limit(5).collect()]
leader_window = Window.partitionBy("community").orderBy(F.desc("pagerank"))
(account_graph_metrics.filter(F.col("community").isin(top_communities))
    .withColumn("rank_in_community", F.row_number().over(leader_window))
    .filter("rank_in_community <= 3")
    .select("community", "rank_in_community", "id", "followers", "in_degree", "pagerank")
    .orderBy("community", "rank_in_community")
    .show(truncate=False))

## 5. Motif finding: reciprocal relationships and broker accounts that connect two others (a → b → c)

In [ ]:
reciprocal = graph.find("(a)-[]->(b); (b)-[]->(a)").filter("a.id < b.id")
n_reciprocal = reciprocal.count()
print(f"reciprocal pairs: {n_reciprocal:,} ({2 * n_reciprocal / n_edges:.2%} of edges are reciprocated)")

brokers = (graph.find("(a)-[]->(b); (b)-[]->(c)")
    .filter("a.id <> c.id")
    .groupBy(F.col("b.id").alias("broker"))
    .agg(F.count("*").alias("paths_through"))
    .orderBy(F.desc("paths_through")))
brokers.show(10, truncate=False)

In [ ]:
from pyspark import StorageLevel

triangles = graph.triangleCount(storage_level=StorageLevel.MEMORY_AND_DISK)
print(f"triangles: {triangles.agg(F.sum('count')).first()[0] // 3:,}")
triangles.orderBy(F.desc("count")).select("id", "count").show(5, truncate=False)

## 6. Outreach simulation: share of the network directly reached by 20 seed accounts, chosen by PageRank vs by follower count

In [ ]:
def one_hop_reach(seed_ids):
    seeds = spark.createDataFrame([(s,) for s in seed_ids], "id string")
    reached = (edges.join(seeds.withColumnRenamed("id", "src"), "src").select(F.col("dst").alias("id"))
        .union(edges.join(seeds.withColumnRenamed("id", "dst"), "dst").select(F.col("src").alias("id")))
        .union(seeds)
        .distinct())
    return reached.count()

active_accounts = edges.select(F.col("src").alias("id")).union(edges.select(F.col("dst").alias("id"))).distinct().count()
seed_strategies = {
    "top 20 by PageRank": [r["id"] for r in ranked.orderBy(F.desc("pagerank")).limit(20).collect()],
    "top 20 by followers": [r["id"] for r in ranked.orderBy(F.desc_nulls_last("followers")).limit(20).collect()],
}
reach = pd.DataFrame([{"strategy": name, "accounts_reached": one_hop_reach(ids)} for name, ids in seed_strategies.items()])
reach["share_of_active_accounts"] = (reach["accounts_reached"] / active_accounts).round(4)
reach

In [ ]:
pr_reach, follower_reach = reach["accounts_reached"].tolist()
print(f"20 PageRank seeds reach {pr_reach:,} accounts directly vs {follower_reach:,} for the 20 largest audiences "
      f"({pr_reach / follower_reach:.2f}x); the largest component holds {giant / n_accounts:.1%} of accounts.")